# EARL shape matching: sessions per user grow step by step

Notebook version of `experiment_earl/src/00–05`. It follows the whiteboard in `image.png`: a fixed set of users
(legitimate **L** and impostor **I**), and the number of sessions per user grows step by step (1, 2, ... up to the maximum).
After every step the result is recorded.

**Nothing is hardcoded here: every setting comes from `config.yaml`.** The number of legitimate users is `n_legitimate`,
of impostor users `n_impostor`, the session steps run from 1 to `sessions_per_user`, and every step is repeated for `n_trials`
random draws. Change the config and re-run: a new config gets its own folder under `runs/`.

**All of Balabit is used** (adviser, 2026-10-08). Who is genuine or impostor comes from Balabit's labels (`labels_file`, `is_illegal`),
switched on per label in `config.yaml`:
- `test_genuine_sessions`: test sessions labeled genuine join their user's sessions, after the training ones, so `sessions_per_user` can go up to 30.
- `test_impostor_attempts`: test sessions labeled impostor are Balabit's own impostor attempts (someone else posing as the folder's user);
  each step checks how many of them the legitimate users' profiles accept.
- Unlabeled test sessions are always left out, because their user is unknown.

Pipeline (Balabit training files only):

1. Draw users and sessions. The sessions of step *k+1* contain the sessions of step *k*.
2. Split each session into chunks at pauses (`pause_gap_s`) and at every click.
3. Shift each chunk to the origin, scale its bounding-box diagonal to 1, resample it to `resample_n` points by arc length.
4. A chunk *matches* when a chunk of the same kind (move/drag) in the legitimate library is within banded DTW cost `dtw_tolerance`.
5. A session matches a profile when at least `match_shape_ratio` of its chunks match; an impostor user is *matched* when at least
   `match_session_ratio` of their sessions match **any single** legitimate profile.
6. `threshold = matched impostor users / total impostor users` (0 = best: impostors never look like the legitimate users).

Run with the repo's `.venv` kernel (it has numba). Results and figures go to this config's folder under `runs/`; `src/08_run_notebook.py` also saves an executed copy of this notebook there.

In [ ]:
import json
import random
import sys
from importlib import import_module
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numba import njit, prange

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()  # experiment_earl/
sys.path.insert(0, str(ROOT / "src"))
cfg_mod = import_module("00_config")

# Every config gets its own folder under runs/ (the same settings always map to the same folder).
# Set the EARL_RUN environment variable to a folder name to rerun an older run instead.
config, RUN = cfg_mod.open_run()
print("run folder:", RUN)

# What this notebook tests: all of it comes from config.yaml
N_LEGITIMATE = config["n_legitimate"]
N_IMPOSTOR = config["n_impostor"]
SESSION_STEPS = list(range(1, config["sessions_per_user"] + 1))  # 1, 2, ... sessions_per_user
LEGIT_STEPS = list(range(1, N_LEGITIMATE + 1))                   # heatmap columns: first 1, 2, ... legitimate users
N_TRIALS = config["n_trials"]                                    # random draws, to see how stable each step is
SEED = config["seed"]
print(f"{N_LEGITIMATE} legitimate + {N_IMPOSTOR} impostor users, sessions {SESSION_STEPS}, {N_TRIALS} draws, seed {SEED}")

print({k: config[k] for k in ["pause_gap_s", "min_points", "min_length_px", "resample_n",
                              "dtw_band", "dtw_tolerance", "match_shape_ratio", "match_session_ratio"]})

## 1. Load a session and split it into chunks

In [ ]:
INVALID_COORDINATE = 65535  # Balabit logging glitch, not a position
MOVE, DRAG = 0, 1
DATASET = ROOT / config["dataset_dir"]


def load_session(path: Path) -> pd.DataFrame:
    """Columns t (seconds from the first event), x, y, state, in time order.

    Scroll rows carry no path and glitch rows (x or y = 65535) are not positions, so both are dropped.
    The sort is stable because many Balabit timestamps tie.
    """
    df = pd.read_csv(path, usecols=["client timestamp", "button", "state", "x", "y"])
    df = df.rename(columns={"client timestamp": "t"}).dropna(subset=["t", "x", "y"])
    df = df[(df["button"] != "Scroll") & (df["x"] < INVALID_COORDINATE) & (df["y"] < INVALID_COORDINATE)]
    df = df.sort_values("t", kind="stable").reset_index(drop=True)
    df["t"] = df["t"] - df["t"].iloc[0]
    return df[["t", "x", "y", "state"]]


def path_length(chunk: np.ndarray) -> float:
    return float(np.hypot(*np.diff(chunk[:, :2], axis=0).T).sum())


def segment(df: pd.DataFrame) -> list[tuple[np.ndarray, int]]:
    """Raw chunks as ([x, y, t] array, kind). A new chunk starts after a pause or any click event."""
    click = df["state"].isin(["Pressed", "Released"]).to_numpy()
    clicks_before = np.cumsum(click)
    keep = ~click
    t = df["t"].to_numpy()[keep]
    xyt = np.column_stack([df["x"].to_numpy()[keep], df["y"].to_numpy()[keep], t])
    drag = df["state"].to_numpy()[keep] == "Drag"
    clicked = np.diff(clicks_before[keep], prepend=clicks_before[keep][0]) > 0
    new_chunk = (np.diff(t, prepend=t[0]) > config["pause_gap_s"]) | clicked
    starts = np.flatnonzero(new_chunk)
    starts = np.concatenate([[0], starts[starts > 0], [len(t)]])
    chunks = []
    for lo, hi in zip(starts[:-1], starts[1:]):
        if hi - lo >= config["min_points"] and path_length(xyt[lo:hi]) >= config["min_length_px"]:
            chunks.append((xyt[lo:hi], DRAG if drag[lo:hi].mean() > 0.5 else MOVE))
    return chunks


def normalize(chunk: np.ndarray, n: int) -> np.ndarray | None:
    """Start at the origin, bounding-box diagonal 1, n points evenly spaced by arc length."""
    xy = chunk[:, :2] - chunk[0, :2]
    diag = np.hypot(np.ptp(xy[:, 0]), np.ptp(xy[:, 1]))
    arc = np.concatenate([[0.0], np.cumsum(np.hypot(*np.diff(xy, axis=0).T))])
    if diag == 0 or arc[-1] == 0:
        return None
    grid = np.linspace(0, arc[-1], n)
    return np.column_stack([np.interp(grid, arc, xy[:, 0]), np.interp(grid, arc, xy[:, 1])]) / diag


class Session:
    """n_chunks counts every recorded chunk (also ones too flat to compare); paths holds the comparable ones."""

    def __init__(self, key: str, chunks: list[tuple[np.ndarray, int]]):
        n = config["resample_n"]
        self.key, self.n_chunks, self.paths = key, len(chunks), {}
        for kind in (MOVE, DRAG):
            paths = [normalize(c, n) for c, k in chunks if k == kind]
            self.paths[kind] = np.array([p for p in paths if p is not None]).reshape(-1, n, 2)


_SESSIONS: dict[str, Session] = {}


def get_session(user: str, name: str, folder: str = config["dataset_dir"]) -> Session:
    """A session from `folder` (training files by default, or test files)."""
    key = f"{folder}/{user}/{name}"
    if key not in _SESSIONS:
        _SESSIONS[key] = Session(key, segment(load_session(ROOT / folder / user / name)))
    return _SESSIONS[key]


demo = get_session("user7", sorted(p.name for p in (DATASET / "user7").iterdir())[0])
print(demo.key, "->", demo.n_chunks, "chunks;", {("move", "drag")[k]: len(v) for k, v in demo.paths.items()}, "comparable")

## 2. Match chunks with banded DTW

The DTW gives up as soon as every cell of a row exceeds the tolerance, which keeps millions of comparisons fast.

In [ ]:
@njit(cache=True)
def _dtw_within(a, b, band, cap, prev, cur):
    """True if the banded DTW cost between paths a and b is at most `cap`."""
    n = a.shape[0]
    prev[:] = np.inf
    prev[0] = 0.0
    for i in range(n):
        cur[:] = np.inf
        row_min = np.inf
        for j in range(max(0, i - band), min(n - 1, i + band) + 1):
            dx, dy = a[i, 0] - b[j, 0], a[i, 1] - b[j, 1]
            value = np.sqrt(dx * dx + dy * dy) + min(prev[j], prev[j + 1], cur[j])
            cur[j + 1] = value
            row_min = min(row_min, value)
        if row_min > cap:
            return False
        prev, cur = cur, prev
    return prev[n] <= cap


@njit(parallel=True, cache=True)
def has_match(queries, library, band, cap):
    """Per query path: does any library path lie within DTW cost `cap`?"""
    n = queries.shape[1]
    found = np.zeros(queries.shape[0], dtype=np.bool_)
    for q in prange(queries.shape[0]):
        prev, cur = np.empty(n + 1), np.empty(n + 1)
        for k in range(library.shape[0]):
            if _dtw_within(queries[q], library[k], band, cap, prev, cur):
                found[q] = True
                break
    return found


_PAIRS: dict = {}  # (query session, library session) -> {kind: bool per query chunk}


def pair_matches(query: Session, library: Session) -> dict[int, np.ndarray]:
    key = (query.key, library.key)
    if key not in _PAIRS:
        n = config["resample_n"]
        band, cap = int(round(config["dtw_band"] * n)), config["dtw_tolerance"] * n
        _PAIRS[key] = {
            kind: has_match(paths, library.paths[kind], band, cap) if len(library.paths[kind])
            else np.zeros(len(paths), dtype=bool)
            for kind, paths in query.paths.items()
        }
    return _PAIRS[key]


def match_ratio(query: Session, library: list[Session]) -> float:
    """Share of the query session's chunks (same kind only) that match a chunk in any library session."""
    matched = 0
    for kind, paths in query.paths.items():
        found = np.zeros(len(paths), dtype=bool)
        for session in library:
            found |= pair_matches(query, session)[kind]
        matched += int(found.sum())
    return matched / max(query.n_chunks, 1)

## 3. Draw users and sessions

The users are drawn once per trial (`n_legitimate` legitimate, `n_impostor` impostor, no overlap). Each user's session pool (`00_config.session_pool`:
training sessions shuffled, then genuine test sessions shuffled) is fixed per draw, and step *k* takes the first *k*, so the 2-session profile contains
the 1-session profile and the 3-session profile contains both. Training sessions come first because they are long (about 2 hours) and test sessions short
(about 5 minutes); with 5–7 training sessions per user, steps beyond that add test sessions.

In [ ]:
ALL_USERS = cfg_mod.list_users(config)
print(cfg_mod.describe_data(config))

fewest_sessions = cfg_mod.max_sessions(config)
if N_LEGITIMATE + N_IMPOSTOR > len(ALL_USERS):
    raise SystemExit(f"{N_LEGITIMATE} + {N_IMPOSTOR} users do not fit in the {len(ALL_USERS)} Balabit users; lower n_legitimate or n_impostor in config.yaml")
if max(SESSION_STEPS) > fewest_sessions:
    raise SystemExit(f"sessions_per_user = {max(SESSION_STEPS)} but some user has only {fewest_sessions} sessions; lower it in config.yaml")


def draw_trial(seed: int) -> tuple[list[str], list[str], dict[str, list[tuple[str, str]]]]:
    """Legitimate and impostor users, and each one's shuffled session pool as (folder, name)."""
    rng = random.Random(seed)
    legit = rng.sample(ALL_USERS, N_LEGITIMATE)  # kept in draw order so legit[:k] is a random subset
    impostor = sorted(rng.sample([u for u in ALL_USERS if u not in legit], N_IMPOSTOR))
    order = {user: cfg_mod.session_pool(config, user, seed) for user in legit + impostor}
    return legit, impostor, order


legit, impostor, order = draw_trial(SEED)
print("legitimate:", legit)
print("impostor:  ", impostor)
pd.DataFrame({u: pd.Series([f"{'train' if f == config['dataset_dir'] else 'test'}: {n}" for f, n in pool])
              for u, pool in order.items()}).head(max(SESSION_STEPS))

## 4. Run one trial and record each step

For every session count, each impostor is compared with each legitimate profile separately. The impostor is matched if any single profile reaches
`match_session_ratio`.

`attempt_step` checks Balabit's own impostor attempts: every test session labeled impostor in a legitimate user's folder is scored against that user's
profile, and it is accepted when at least `match_shape_ratio` of its chunks match (the same rule as one session in `run_step`).

In [ ]:
def run_step(legit, impostor, order, n_sessions) -> list[dict]:
    pick = lambda user: [get_session(user, name, folder) for folder, name in order[user][:n_sessions]]
    profiles = {user: pick(user) for user in legit}
    rows = []
    for user in impostor:
        sessions = pick(user)
        per_profile = {p: [match_ratio(s, library) for s in sessions] for p, library in profiles.items()}
        share = {p: float(np.mean([r >= config["match_shape_ratio"] for r in ratios])) for p, ratios in per_profile.items()}
        best = max(per_profile, key=lambda p: (share[p], np.mean(per_profile[p])))
        rows.append({
            "sessions": n_sessions, "impostor": user, "best_profile": best,
            "mean_chunk_ratio": round(float(np.mean(per_profile[best])), 3),
            "sessions_matched": f"{round(share[best] * len(sessions))}/{len(sessions)}",
            "matched": share[best] >= config["match_session_ratio"],
        })
    return rows


def attempt_step(legit, order, n_sessions) -> list[dict]:
    """Balabit impostor attempts against each legitimate user's own profile (empty unless test_impostor_attempts)."""
    rows = []
    for user in legit:
        attempts = cfg_mod.impostor_attempts(config, user)
        if not attempts:
            continue
        library = [get_session(user, name, folder) for folder, name in order[user][:n_sessions]]
        ratios = [match_ratio(get_session(user, name, config["test_dir"]), library) for name in attempts]
        accepted = sum(r >= config["match_shape_ratio"] for r in ratios)
        rows.append({"sessions": n_sessions, "legitimate": user, "attempts": len(ratios),
                     "accepted": accepted, "accept_rate": accepted / len(ratios)})
    return rows


detail = pd.DataFrame([row for k in SESSION_STEPS for row in run_step(legit, impostor, order, k)])
attempt_detail = pd.DataFrame([row for k in SESSION_STEPS for row in attempt_step(legit, order, k)])
display(attempt_detail) if len(attempt_detail) else print("test_impostor_attempts is false: no Balabit impostor attempts")
detail

In [ ]:
def threshold_table(detail: pd.DataFrame) -> pd.DataFrame:
    table = detail.groupby("sessions").agg(matched_detected=("matched", "sum"), total_impostor_user=("matched", "size"))
    table["threshold"] = table["matched_detected"] / table["total_impostor_user"]
    return table


threshold_table(detail)

## 5. Repeat over random draws

One draw can only give 0 to `n_impostor` matched impostors, so the same experiment is repeated for `n_trials` seeds (`SEED`, `SEED + 1`, ...).
Every draw is also scored with only the first 1 and 2 of its legitimate users, which gives the heatmap columns.

Recorded in the run folder (`runs/<run>/results/`):
- `session_progression_trials.csv`: every impostor, every step
- `session_progression.csv`: one row per session step (all `n_legitimate` legitimate users)
- `session_progression_heatmap.csv`: mean threshold per (sessions, legitimate users)
- `session_progression_attempts.csv`: Balabit impostor attempts per draw, step and legitimate user (with `test_impostor_attempts`)

In [ ]:
rows, attempt_rows = [], []
for trial in range(N_TRIALS):
    legit_t, impostor_t, order_t = draw_trial(SEED + trial)
    for k in SESSION_STEPS:
        attempt_rows += [{"trial": trial, "seed": SEED + trial, **row} for row in attempt_step(legit_t, order_t, k)]
        for n_legit in LEGIT_STEPS:
            rows += [{"trial": trial, "seed": SEED + trial, "legit_users": n_legit,
                      "legitimate": ",".join(legit_t[:n_legit]), **row}
                     for row in run_step(legit_t[:n_legit], impostor_t, order_t, k)]
trials = pd.DataFrame(rows)

per_trial = (trials.groupby(["legit_users", "sessions", "trial"])["matched"]
             .agg(matched_detected="sum", threshold="mean").reset_index())

full = per_trial[per_trial["legit_users"] == N_LEGITIMATE]
summary = full.groupby("sessions").agg(
    matched_detected_mean=("matched_detected", "mean"),
    matched_detected_std=("matched_detected", "std"),
    threshold_mean=("threshold", "mean"),
)
attempts = pd.DataFrame(attempt_rows)
if len(attempts):
    # share of all attempts on this draw's legitimate users that were accepted, averaged over the draws
    per_draw = attempts.groupby(["sessions", "trial"])[["accepted", "attempts"]].sum()
    summary["attempt_accept_rate_mean"] = (per_draw["accepted"] / per_draw["attempts"]).groupby("sessions").mean()
summary["n_legit"], summary["n_impostor"], summary["n_trials"] = N_LEGITIMATE, N_IMPOSTOR, N_TRIALS
summary = summary.round(3)

heatmap = per_trial.pivot_table(index="sessions", columns="legit_users", values="threshold", aggfunc="mean")
heatmap_matched = per_trial.pivot_table(index="sessions", columns="legit_users", values="matched_detected", aggfunc="mean")

(RUN / "results").mkdir(exist_ok=True)
trials.to_csv(RUN / "results" / "session_progression_trials.csv", index=False)
summary.to_csv(RUN / "results" / "session_progression.csv")
heatmap.round(3).to_csv(RUN / "results" / "session_progression_heatmap.csv")
if len(attempts):
    attempts.to_csv(RUN / "results" / "session_progression_attempts.csv", index=False)
summary

## 6. Bar graph: impostors matched per session step

All `n_legitimate` legitimate users; the bar is the mean number of impostor users (out of `n_impostor`) that matched, over the draws.
With `test_impostor_attempts`, the right panel shows the share of Balabit impostor attempts the profiles accepted (lower is better).

In [ ]:
with_attempts = "attempt_accept_rate_mean" in summary
fig, axes = plt.subplots(1, 2 if with_attempts else 1, figsize=(11 if with_attempts else 5.5, 4), squeeze=False)
ax = axes[0][0]
bars = ax.bar([str(k) for k in summary.index], summary["matched_detected_mean"], color="tab:blue")
ax.bar_label(bars, labels=[f"{v:.2f}" for v in summary["matched_detected_mean"]], padding=3)
ax.set_ylim(0, N_IMPOSTOR * 1.15)  # headroom for the value labels
ax.set_yticks(range(N_IMPOSTOR + 1))
ax.set_xlabel("sessions per user")
ax.set_ylabel(f"impostors matched (out of {N_IMPOSTOR})")
draws = "1 draw" if N_TRIALS == 1 else f"mean of {N_TRIALS} draws"
ax.set_title(f"{N_LEGITIMATE} legitimate + {N_IMPOSTOR} impostor users, {draws}")
if with_attempts:
    ax = axes[0][1]
    bars = ax.bar([str(k) for k in summary.index], summary["attempt_accept_rate_mean"] * 100, color="tab:red")
    ax.bar_label(bars, labels=[f"{v:.0%}" for v in summary["attempt_accept_rate_mean"]], padding=3)
    ax.set_ylim(0, 115)
    ax.set_xlabel("sessions per user")
    ax.set_ylabel("Balabit impostor attempts accepted (%)")
    ax.set_title(f"Balabit impostor attempts, {draws}")
fig.tight_layout()
(RUN / "figures").mkdir(exist_ok=True)
fig.savefig(RUN / "figures" / "session_progression.png", dpi=150)
plt.show()

## 7. Heatmap: sessions per user x legitimate users

Cell colour = mean threshold (impostors matched / `n_impostor`), fixed 0.0 to 1.0 scale in steps of 0.1, 0 = best.
The second line in each cell is the same result as the mean number of impostors matched.

In [ ]:
from matplotlib.colors import BoundaryNorm

levels = np.round(np.arange(0, 1.01, 0.1), 1)
fig, ax = plt.subplots(figsize=(6, 4.5))
image = ax.imshow(heatmap.to_numpy(), cmap="YlOrRd", norm=BoundaryNorm(levels, 256), origin="lower", aspect="auto")
ax.set_xticks(range(len(heatmap.columns)), heatmap.columns)
ax.set_yticks(range(len(heatmap.index)), heatmap.index)
for r, k in enumerate(heatmap.index):
    for c, n_legit in enumerate(heatmap.columns):
        ax.text(c, r, f"{heatmap.loc[k, n_legit]:.2f}\n({heatmap_matched.loc[k, n_legit]:.2f} of {N_IMPOSTOR})",
                ha="center", va="center", fontsize=9)
ax.set_xlabel("number of legitimate users")
ax.set_ylabel("number of sessions per user")
ax.set_title(f"{'Threshold' if N_TRIALS == 1 else 'Mean threshold'}, {N_IMPOSTOR} impostor users, {draws}")
fig.colorbar(image, ax=ax, ticks=levels, label="threshold (0 = best)")
fig.tight_layout()
fig.savefig(RUN / "figures" / "session_progression_heatmap.png", dpi=150)
plt.show()

## Notes

- The tolerance in `config.yaml` (0.02) is the value that keeps the metric from being all 0 or all 1. In the earlier sweep a user's own held-out sessions never matched
  more than impostors did, so read these numbers as a baseline for how the threshold grows with more sessions, not as proof that shapes are unique to a user.
- Balabit has 10 users, so `n_legitimate + n_impostor` can be at most 10, and `sessions_per_user` at most 5 (the smallest session count of any user); the notebook stops with a message otherwise.
- Settings come from `config.yaml`; a new config creates a new `runs/` folder, so older results are kept.
- The `src/` scripts remain the reference for the file-based flow (`temp/`, `shapes/`, `results/trial_k.json`).